# Kon-Tiki Biochar Volume — Web Dashboard (upload video → volume)

**One step:** `Runtime → Change runtime type → GPU (T4)`, then **Run this cell**.
After ~1–2 min it prints a **public link** (`…gradio.live`). Open it → **upload a kiln
video → get the volume**. No download / re-upload. Share the link with anyone.

_The GPU reconstruction runs here on Colab's free GPU (this is the only part that needs a
GPU); the volume maths is the same code validated to ~2–4% on ground truth._


In [ ]:
import os, sys, glob, shutil, base64, subprocess, torch, numpy as np, cv2
subprocess.run("pip -q install gradio opencv-python-headless scipy", shell=True)
if not os.path.exists("vggt"):
    subprocess.run("git clone -q https://github.com/facebookresearch/vggt.git", shell=True)
subprocess.run("grep -viE '^(torch|torchvision|torchaudio|numpy)' vggt/requirements.txt > /tmp/r.txt", shell=True)
subprocess.run("pip -q install -r /tmp/r.txt", shell=True)
if "vggt" not in sys.path: sys.path.append("vggt")
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU (T4), then re-run."

open("estimate_volume.py", "w", encoding="utf-8").write(base64.b64decode("IiIiVmlkZW8gLT4gVm9sdW1lIHBpcGVsaW5lLCBWT0xVTUUgU1RFUCAobG9jYWwsIENQVSDigJQgbm8gR1BVIG5lZWRlZCkuCgpUYWtlcyBhIDMtRCBwb2ludCBjbG91ZCBvZiBhIGJpb2NoYXItZmlsbGVkIEtvbi1UaWtpIGtpbG4gKGZyb20gdGhlIHJlY29uc3RydWN0aW9uCnN0ZXApIGFuZCByZXR1cm5zIHRoZSBiaW9jaGFyIHZvbHVtZSBpbiBsaXRyZXMuIFB1cmUgZ2VvbWV0cnk6CiAgMS4gZmluZCAndXAnIGZyb20gdGhlIGRvbWluYW50IHBsYW5lOyBwdXQgdGhlICp3aWRlc3QqIGhvcml6b250YWwgc2hlZXQgKGdyb3VuZCkKICAgICBhdCB0aGUgYm90dG9tICAoc28gd2UgbmV2ZXIgY29uZnVzZSB0aGUgYmlvY2hhciBzdXJmYWNlIGZvciB0aGUgZ3JvdW5kKSwKICAyLiBpc29sYXRlIHRoZSBraWxuLCBmaXQgdGhlIHJpbSAtPiBzY2FsZSB0aGUgY2xvdWQgdG8gcmVhbCBjbSAocmltIHJhZGl1cyA3NSBjbSksCiAgMy4gaW50ZWdyYXRlIHRoZSBtZWFzdXJlZCBiaW9jaGFyIHN1cmZhY2UgYWdhaW5zdCB0aGUga25vd24ga2lsbiBjb25lLCBmaWxsaW5nCiAgICAgZ2FwcyBieSBuZWFyZXN0LW5laWdoYm91ciBzbyBzcGFyc2Ugc3BvdHMgZG9uJ3QgdW5kZXItY291bnQuCgpTYW1lIGxvZ2ljIHRoZSBDb2xhYiBub3RlYm9vayB1c2VzOyBpdCBydW5zIGhlcmUgb24gQ1BVIGJlY2F1c2UgaXQgaXMgbm90IEdQVSB3b3JrLgoKVXNhZ2U6ICBweXRob24gZXN0aW1hdGVfdm9sdW1lLnB5IGNsb3VkLnBseSBbcmltX3JhZGl1c19jbV0gW3ZpZXdzLnBuZ10KIiIiCmltcG9ydCBzeXMsIG51bXB5IGFzIG5wCmZyb20gc2NpcHkuaW50ZXJwb2xhdGUgaW1wb3J0IE5lYXJlc3ROREludGVycG9sYXRvcgpmcm9tIHNjaXB5LnNwYXRpYWwgaW1wb3J0IGNLRFRyZWUKCiMgS29uLVRpa2kgMTAwMCBnZW9tZXRyeSAoY20pLCBmcm9tIHRoZSBkZXNpZ24gZHJhd2luZwpSX0NNLCBSQl9DTSwgSF9DTSA9IDc1LjAsIDQxLjE1LCA5My4wICAgIyByaW0gw5gxNTAwLCBib3R0b20gw5g4MjMsIGRlcHRoIDkzMCAoZGVzaWduIGRyYXdpbmcpCkRFTlNJVFkgPSAwLjI1ICAjIGtnIC8gTApDRUxMID0gMy4wICAgICAgIyBpbnRlZ3JhdGlvbiBncmlkIChjbSkKVE9QX1BDVCA9IDEyICAgICMgcGVyLWNlbGwgcGVyY2VudGlsZSA9IHRoZSB0b3AgKGJpb2NoYXIpIHN1cmZhY2UsIHJvYnVzdCB0byBkZWVwIGFydGVmYWN0cwpDT0xfTUlOID0gOS4wICAgIyBjbTogbWluIGJpb2NoYXIgY29sdW1uIHRvIGNvdW50IChyZWplY3RzIHRoZSBzdGVlcC13YWxsIHJpbmc7IH5DRUxMKkgvKFItUkIpKQoKCmRlZiByb3RfZnJvbV90byhhLCBiKToKICAgIGEgPSBhIC8gbnAubGluYWxnLm5vcm0oYSk7IGIgPSBiIC8gbnAubGluYWxnLm5vcm0oYikKICAgIHYgPSBucC5jcm9zcyhhLCBiKTsgYyA9IGZsb2F0KG5wLmRvdChhLCBiKSkKICAgIGlmIG5wLmxpbmFsZy5ub3JtKHYpIDwgMWUtODoKICAgICAgICByZXR1cm4gbnAuZXllKDMpIGlmIGMgPiAwIGVsc2UgbnAuZGlhZyhbMS4wLCAtMS4wLCAtMS4wXSkKICAgIHZ4ID0gbnAuYXJyYXkoW1swLCAtdlsyXSwgdlsxXV0sIFt2WzJdLCAwLCAtdlswXV0sIFstdlsxXSwgdlswXSwgMF1dKQogICAgcmV0dXJuIG5wLmV5ZSgzKSArIHZ4ICsgdnggQCB2eCAqICgxLjAgLyAoMS4wICsgYykpCgoKZGVmIGZpdF9jaXJjbGUoeHkpOgogICAgeCwgeSA9IHh5WzosIDBdLCB4eVs6LCAxXQogICAgQSA9IG5wLmNfWzIgKiB4LCAyICogeSwgbnAub25lcyhsZW4oeCkpXTsgYiA9IHggKiogMiArIHkgKiogMgogICAgYywgKl8gPSBucC5saW5hbGcubHN0c3EoQSwgYiwgcmNvbmQ9Tm9uZSkKICAgIGN4LCBjeSA9IGNbMF0sIGNbMV0KICAgIHJldHVybiBjeCwgY3ksIG5wLnNxcnQobWF4KGNbMl0gKyBjeCAqKiAyICsgY3kgKiogMiwgMWUtOSkpCgoKZGVmIF9zZWdtZW50X3BsYW5lKFAsIHRociwgaXRlcnM9MjAwMCwgc2VlZD0wKToKICAgICIiIk1pbmltYWwgUkFOU0FDIHBsYW5lIGZpdCAtPiAobm9ybWFsLCBpbmxpZXJfbWFzaykuIE5vIG9wZW4zZCBkZXBlbmRlbmN5LiIiIgogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCiAgICBiZXN0X24sIGJlc3RfaW4gPSBOb25lLCBOb25lCiAgICBuX2Jlc3QgPSAwCiAgICBmb3IgXyBpbiByYW5nZShpdGVycyk6CiAgICAgICAgaWR4ID0gcm5nLmNob2ljZShsZW4oUCksIDMsIHJlcGxhY2U9RmFsc2UpCiAgICAgICAgcDAsIHAxLCBwMiA9IFBbaWR4XQogICAgICAgIG5ybSA9IG5wLmNyb3NzKHAxIC0gcDAsIHAyIC0gcDApCiAgICAgICAgbmwgPSBucC5saW5hbGcubm9ybShucm0pCiAgICAgICAgaWYgbmwgPCAxZS05OgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIG5ybSA9IG5ybSAvIG5sCiAgICAgICAgZCA9IG5wLmFicygoUCAtIHAwKSBAIG5ybSkKICAgICAgICBpbmwgPSBkIDwgdGhyCiAgICAgICAgYyA9IGludChpbmwuc3VtKCkpCiAgICAgICAgaWYgYyA+IG5fYmVzdDoKICAgICAgICAgICAgbl9iZXN0LCBiZXN0X24sIGJlc3RfaW4gPSBjLCBucm0sIGlubAogICAgcmV0dXJuIGJlc3RfbiwgYmVzdF9pbgoKCmRlZiBfbGFyZ2VzdF9jbHVzdGVyKFAsIGVwcywgbWluX3B0cz0yMCk6CiAgICAiIiJHcmlkLWJhc2VkIGNvbm5lY3RlZC1jb21wb25lbnRzIGNsdXN0ZXJpbmcgKGZhc3QsIG5vIG9wZW4zZCkuIiIiCiAgICBrZXlzID0gbnAuZmxvb3IoUCAvIGVwcykuYXN0eXBlKG5wLmludDY0KQogICAgZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgZGVmYXVsdGRpY3QKICAgIGNlbGwgPSBkZWZhdWx0ZGljdChsaXN0KQogICAgZm9yIGksIGsgaW4gZW51bWVyYXRlKG1hcCh0dXBsZSwga2V5cykpOgogICAgICAgIGNlbGxba10uYXBwZW5kKGkpCiAgICBzZWVuLCBiZXN0ID0gc2V0KCksIFtdCiAgICBuZWlnaCA9IFsoZHgsIGR5LCBkeikgZm9yIGR4IGluICgtMSwgMCwgMSkgZm9yIGR5IGluICgtMSwgMCwgMSkgZm9yIGR6IGluICgtMSwgMCwgMSldCiAgICBmb3Igc3RhcnQgaW4gY2VsbDoKICAgICAgICBpZiBzdGFydCBpbiBzZWVuOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHN0YWNrLCBjb21wID0gW3N0YXJ0XSwgW10KICAgICAgICBzZWVuLmFkZChzdGFydCkKICAgICAgICB3aGlsZSBzdGFjazoKICAgICAgICAgICAgYyA9IHN0YWNrLnBvcCgpOyBjb21wLmV4dGVuZChjZWxsW2NdKQogICAgICAgICAgICBmb3IgZCBpbiBuZWlnaDoKICAgICAgICAgICAgICAgIG5iID0gKGNbMF0gKyBkWzBdLCBjWzFdICsgZFsxXSwgY1syXSArIGRbMl0pCiAgICAgICAgICAgICAgICBpZiBuYiBpbiBjZWxsIGFuZCBuYiBub3QgaW4gc2VlbjoKICAgICAgICAgICAgICAgICAgICBzZWVuLmFkZChuYik7IHN0YWNrLmFwcGVuZChuYikKICAgICAgICBpZiBsZW4oY29tcCkgPiBsZW4oYmVzdCk6CiAgICAgICAgICAgIGJlc3QgPSBjb21wCiAgICByZXR1cm4gbnAuYXJyYXkoYmVzdCkgaWYgbGVuKGJlc3QpID49IG1pbl9wdHMgZWxzZSBucC5hcmFuZ2UobGVuKFApKQoKCmRlZiBfd2FsbF9kZXB0aChycik6CiAgICAiIiJEZXB0aCAoY20sIGJlbG93IHJpbSkgb2YgdGhlIGtpbG4gd2FsbC9mbG9vciBhdCByYWRpdXMgcnIgKHZlY3RvcmlzZWQpLiIiIgogICAgcmV0dXJuIG5wLndoZXJlKHJyIDw9IFJCX0NNLCBIX0NNLCAoUl9DTSAtIHJyKSAvIChSX0NNIC0gUkJfQ00pICogSF9DTSkKCgpkZWYgZXN0aW1hdGVfcG9pbnRzKFAsIHJpbV9yYWRpdXNfY209Ul9DTSwgdmlld3NfcG5nPU5vbmUsIGhlYXRtYXBfcG5nPU5vbmUsIGRlYnVnPUZhbHNlKToKICAgIFAgPSBucC5hc2FycmF5KFAsIGZsb2F0KQogICAgUCA9IFBbbnAuaXNmaW5pdGUoUCkuYWxsKDEpXQogICAgbWVkID0gbnAubWVkaWFuKFAsIDApOyBkID0gbnAubGluYWxnLm5vcm0oUCAtIG1lZCwgYXhpcz0xKQogICAgUCA9IFBbZCA8IG5wLnBlcmNlbnRpbGUoZCwgOTgpXQogICAgZGlhZyA9IGZsb2F0KG5wLmxpbmFsZy5ub3JtKFAubWF4KDApIC0gUC5taW4oMCkpKQogICAgIyBzY2FsZS1mcmVlIGxvY2FsIHBvaW50IHNwYWNpbmcgKHJvYnVzdCB0byBhIGh1Z2UgZ3JvdW5kIHBsYW5lIGluIHRoZSBzY2VuZSkKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZygwKQogICAgc3ViID0gUFtybmcuY2hvaWNlKGxlbihQKSwgbWluKGxlbihQKSwgNDAwMCksIHJlcGxhY2U9RmFsc2UpXQogICAgc3BhY2luZyA9IGZsb2F0KG5wLm1lZGlhbihjS0RUcmVlKFApLnF1ZXJ5KHN1Yiwgaz0yKVswXVs6LCAxXSkpCgogICAgIyAxKSB1cCBkaXJlY3Rpb24gZnJvbSB0aGUgZG9taW5hbnQgcGxhbmUgKGdyb3VuZCBvciBiaW9jaGFyIHN1cmZhY2UgLT4gc2FtZSBub3JtYWwpCiAgICBuLCBfID0gX3NlZ21lbnRfcGxhbmUoUCwgdGhyPW1heCgyLjUgKiBzcGFjaW5nLCAwLjAwMyAqIGRpYWcpKQogICAgUjEgPSByb3RfZnJvbV90byhuLCBucC5hcnJheShbMCwgMCwgMS4wXSkpCiAgICBRID0gUCBAIFIxLlQKICAgIHogPSBRWzosIDJdOyB6ciA9IHoubWF4KCkgLSB6Lm1pbigpCgogICAgIyAyKSB3aWRlc3QgaG9yaXpvbnRhbCBzbGFiID0gZ3JvdW5kOyBlbnN1cmUgaXQgc2l0cyBhdCB0aGUgYm90dG9tCiAgICBuYiwgYmVzdF93LCBncm91bmRfeiA9IDMwLCAtMSwgTm9uZQogICAgZWRnZXMgPSBucC5saW5zcGFjZSh6Lm1pbigpLCB6Lm1heCgpLCBuYiArIDEpCiAgICBmb3IgaSBpbiByYW5nZShuYik6CiAgICAgICAgbSA9ICh6ID49IGVkZ2VzW2ldKSAmICh6IDwgZWRnZXNbaSArIDFdKQogICAgICAgIGlmIG0uc3VtKCkgPCBtYXgoNTAsIDAuMDA0ICogbGVuKHopKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBjID0gUVttLCA6Ml0ubWVhbigwKQogICAgICAgIHcgPSBucC5wZXJjZW50aWxlKG5wLmh5cG90KFFbbSwgMF0gLSBjWzBdLCBRW20sIDFdIC0gY1sxXSksIDg1KQogICAgICAgIGlmIHcgPiBiZXN0X3c6CiAgICAgICAgICAgIGJlc3RfdywgZ3JvdW5kX3ogPSB3LCAwLjUgKiAoZWRnZXNbaV0gKyBlZGdlc1tpICsgMV0pCiAgICBpZiBncm91bmRfeiBpcyBOb25lOgogICAgICAgIGdyb3VuZF96ID0gei5taW4oKQogICAgZWxpZiBncm91bmRfeiA+IDAuNSAqICh6Lm1pbigpICsgei5tYXgoKSk6CiAgICAgICAgUjEgPSBucC5kaWFnKFsxLjAsIC0xLjAsIC0xLjBdKSBAIFIxICAgICAgICAgICMgZmxpcCAxODAgZGVnIGFib3V0IFgKICAgICAgICBRID0gUCBAIFIxLlQ7IHogPSBRWzosIDJdOyBncm91bmRfeiA9IC1ncm91bmRfegoKICAgICMgMykgZHJvcCB0aGUgZ3JvdW5kIHNoZWV0LCBrZWVwIHRoZSBsYXJnZXN0IGNsdXN0ZXIgKHRoZSBraWxuKQogICAga2lsbiA9IFFbeiA+IGdyb3VuZF96ICsgbWF4KDMgKiBzcGFjaW5nLCAwLjAyICogenIpXQogICAgaWR4ID0gX2xhcmdlc3RfY2x1c3RlcihraWxuLCBlcHM9My4wICogc3BhY2luZykKICAgIEsgPSBraWxuW2lkeF0KCiAgICAjIDNiKSByZWZpbmUgdGhlIGF4aXM6IHRoZSBraWxuIGlzIGEgc3VyZmFjZSBvZiByZXZvbHV0aW9uLCBzbyBpdHMgc3ltbWV0cnkKICAgICMgYXhpcyBpcyB0aGUgc21hbGxlc3QtdmFyaWFuY2UgUENBIGRpcmVjdGlvbiAocm9idXN0IHZzIGEgdGlsdGVkIHBsYW5lIGZpdCkuCiAgICBjMCA9IEsubWVhbigwKQogICAgXywgXywgdnQgPSBucC5saW5hbGcuc3ZkKEsgLSBjMCwgZnVsbF9tYXRyaWNlcz1GYWxzZSkKICAgIGF4aXMgPSB2dFsyXQogICAgaWYgYXhpcyBAIG5wLmFycmF5KFswLCAwLCAxLjBdKSA8IDA6CiAgICAgICAgYXhpcyA9IC1heGlzCiAgICBLID0gKEsgLSBjMCkgQCByb3RfZnJvbV90byhheGlzLCBucC5hcnJheShbMCwgMCwgMS4wXSkpLlQKICAgICMgcmltICh3aWRlIGVuZCkgbXVzdCBiZSBhdCArWjogcmFkaXVzIHNob3VsZCBncm93IHdpdGggaGVpZ2h0CiAgICByaG8gPSBucC5oeXBvdChLWzosIDBdLCBLWzosIDFdKQogICAgaWYgbnAuY29ycmNvZWYoS1s6LCAyXSwgcmhvKVswLCAxXSA8IDA6CiAgICAgICAgS1s6LCAyXSAqPSAtMS4wCiAgICBpZiBkZWJ1ZzoKICAgICAgICBwcmludChmIiAgW2RlYnVnXSBzcGFjaW5nPXtzcGFjaW5nOi4zZn0gbl9raWxuPXtsZW4oSyl9L3tsZW4oa2lsbil9IGF4aXNfej17YXhpc1syXTouM2Z9IikKCiAgICAjIDQpIGZpdCB0aGUga2lsbiBXQUxMIGNvbmUgLT4gcmltIHJhZGl1cyAmIHBsYW5lIC0+IHNjYWxlIHRvIGNtIChheGlzIGF0IG9yaWdpbikuCiAgICAjIFRoZSB3YWxsJ3MgbWF4LXJhZGl1cy12cy1oZWlnaHQgaXMgYSBzdHJhaWdodCBsaW5lOyBleHRyYXBvbGF0ZSB0byB0aGUgdG9wLgogICAgemsgPSBLWzosIDJdCiAgICByaG8gPSBucC5oeXBvdChLWzosIDBdLCBLWzosIDFdKQogICAgemIgPSBucC5saW5zcGFjZSh6ay5taW4oKSwgemsubWF4KCksIDIyKQogICAgenosIHJyID0gW10sIFtdCiAgICBmb3IgaSBpbiByYW5nZShsZW4oemIpIC0gMSk6CiAgICAgICAgbSA9ICh6ayA+PSB6YltpXSkgJiAoemsgPCB6YltpICsgMV0pCiAgICAgICAgaWYgbS5zdW0oKSA8IDIwOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHp6LmFwcGVuZCgwLjUgKiAoemJbaV0gKyB6YltpICsgMV0pKTsgcnIuYXBwZW5kKG5wLnBlcmNlbnRpbGUocmhvW21dLCA5OCkpCiAgICB6eiwgcnIgPSBucC5hcnJheSh6eiksIG5wLmFycmF5KHJyKQogICAgbV9zbG9wZSwgY19pbnQgPSBucC5saW5hbGcubHN0c3EobnAuY19benosIG5wLm9uZXNfbGlrZSh6eildLCByciwgcmNvbmQ9Tm9uZSlbMF0KICAgIHpfcmltID0gZmxvYXQobnAucGVyY2VudGlsZSh6aywgOTkuNSkpCiAgICByX3VuaXRzID0gbV9zbG9wZSAqIHpfcmltICsgY19pbnQKICAgIHMgPSByaW1fcmFkaXVzX2NtIC8gcl91bml0cwogICAgaWYgZGVidWc6CiAgICAgICAgcHJpbnQoZiIgIFtkZWJ1Z10gc2xvcGU9e21fc2xvcGU6LjNmfSByX3VuaXRzPXtyX3VuaXRzOi4zZn0gcz17czouNGZ9IikKICAgIEsgPSAoSyAtIG5wLmFycmF5KFswLjAsIDAuMCwgel9yaW1dKSkgKiBzCgogICAgIyA1KSBUT1Atc3VyZmFjZSBoZWlnaHRtYXAgb3ZlciB0aGUgcmltIGRpc2ssIGludGVncmF0ZWQgYWdhaW5zdCB0aGUga25vd24gY29uZS4KICAgICMgICAgUGVyIGNlbGwgdGFrZSB0aGUgU0hBTExPV0VTVCBwb2ludHMgKHRoZSBiaW9jaGFyIHRvcCkgLT4gaWdub3JlcyBkZWVwCiAgICAjICAgIGludGVyaW9yIC8gcmVjb25zdHJ1Y3Rpb24gYXJ0ZWZhY3RzLCBhbmQgd29ya3MgZm9yIGFueSBmaWxsIGxldmVsLgogICAgZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgZGVmYXVsdGRpY3QKICAgIHgsIHksIHpjID0gS1s6LCAwXSwgS1s6LCAxXSwgS1s6LCAyXQogICAgZGVwID0gLXpjOyByaG8gPSBucC5oeXBvdCh4LCB5KQogICAgVl9mdWxsID0gKDEgLyAzKSAqIG5wLnBpICogSF9DTSAqIChSQl9DTSAqKiAyICsgUkJfQ00gKiBSX0NNICsgUl9DTSAqKiAyKSAvIDEwMDAuMAogICAgY29sZ3JpZCA9IE5vbmUgICAgICAgICAgICAgICAgICAgICMgYmlvY2hhci1kZXB0aCBoZWF0bWFwIChmaWxsZWQgaW4gYmVsb3cpCgogICAgaW5zID0gcmhvIDw9IFJfQ00KICAgIGd4ID0gbnAuZmxvb3IoKHhbaW5zXSArIFJfQ00pIC8gQ0VMTCkuYXN0eXBlKGludCkKICAgIGd5ID0gbnAuZmxvb3IoKHlbaW5zXSArIFJfQ00pIC8gQ0VMTCkuYXN0eXBlKGludCkKICAgIGRlcGkgPSBkZXBbaW5zXQogICAgYWNjID0gZGVmYXVsdGRpY3QobGlzdCkKICAgIGZvciB4aSwgeWksIGRwIGluIHppcChneCwgZ3ksIGRlcGkpOgogICAgICAgIGFjY1soeGksIHlpKV0uYXBwZW5kKGRwKQogICAgY2VsbHMsIGRlcHRocyA9IFtdLCBbXQogICAgZm9yIGtleSwgdiBpbiBhY2MuaXRlbXMoKToKICAgICAgICBpZiBsZW4odikgPCAzOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGNlbGxzLmFwcGVuZChrZXkpOyBkZXB0aHMuYXBwZW5kKG5wLnBlcmNlbnRpbGUodiwgVE9QX1BDVCkpICAgIyB0b3AgPSBiaW9jaGFyIHN1cmZhY2UKICAgIGlmIGxlbihjZWxscykgPCAzMDogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBlc3NlbnRpYWxseSBlbXB0eSBraWxuCiAgICAgICAgVl9MID0gVl9zaW1wbGUgPSBoX2ZpbGwgPSAwLjAKICAgIGVsc2U6CiAgICAgICAgY2VsbHMgPSBucC5hcnJheShjZWxscyk7IGRlcHRocyA9IG5wLmFycmF5KGRlcHRocykKICAgICAgICBjZW50ZXJzID0gKGNlbGxzICsgMC41KSAqIENFTEwgLSBSX0NNCiAgICAgICAgaW50ZXJwID0gTmVhcmVzdE5ESW50ZXJwb2xhdG9yKGNlbnRlcnMsIGRlcHRocykKICAgICAgICBuY2VsbCA9IGludChucC5jZWlsKDIgKiBSX0NNIC8gQ0VMTCkpCiAgICAgICAgY2MgPSAobnAuYXJhbmdlKG5jZWxsKSArIDAuNSkgKiBDRUxMIC0gUl9DTQogICAgICAgIFhYLCBZWSA9IG5wLm1lc2hncmlkKGNjLCBjYyk7IFJSID0gbnAuaHlwb3QoWFgsIFlZKQogICAgICAgIGRpc2sgPSBSUiA8PSBSX0NNCiAgICAgICAgZHN1cmYgPSBpbnRlcnAoWFhbZGlza10sIFlZW2Rpc2tdKQogICAgICAgIGNvbCA9IF93YWxsX2RlcHRoKFJSW2Rpc2tdKSAtIGRzdXJmCiAgICAgICAgVl9MID0gZmxvYXQoY29sW2NvbCA+IENPTF9NSU5dLnN1bSgpICogQ0VMTCAqIENFTEwgLyAxMDAwLjApCiAgICAgICAgY2cgPSBfd2FsbF9kZXB0aChSUikgLSBpbnRlcnAoWFgsIFlZKSAgICAgICAgICAjIGZ1bGwtZ3JpZCBiaW9jaGFyIGRlcHRoIGhlYXRtYXAKICAgICAgICBjb2xncmlkID0gbnAud2hlcmUoZGlzayAmIChjZyA+IENPTF9NSU4pLCBjZywgbnAubmFuKQogICAgICAgICMgZmxhdCBjcm9zcy1jaGVjayBmcm9tIHRoZSBjZWxscyB0aGF0IGFjdHVhbGx5IGhvbGQgYmlvY2hhcgogICAgICAgIGNvbGMgPSBfd2FsbF9kZXB0aChucC5oeXBvdChjZW50ZXJzWzosIDBdLCBjZW50ZXJzWzosIDFdKSkgLSBkZXB0aHMKICAgICAgICBiaW9fZCA9IGRlcHRoc1tjb2xjID4gQ09MX01JTl0KICAgICAgICBkX21lZCA9IGZsb2F0KG5wLm1lZGlhbihiaW9fZCkpIGlmIGxlbihiaW9fZCkgZWxzZSBmbG9hdChucC5tZWRpYW4oZGVwdGhzKSkKICAgICAgICBoX2ZpbGwgPSBmbG9hdChucC5jbGlwKEhfQ00gLSBkX21lZCwgMCwgSF9DTSkpCiAgICAgICAgcnMgPSBSQl9DTSArIChSX0NNIC0gUkJfQ00pICogKGhfZmlsbCAvIEhfQ00pCiAgICAgICAgVl9zaW1wbGUgPSAoMSAvIDMpICogbnAucGkgKiBoX2ZpbGwgKiAoUkJfQ00gKiogMiArIFJCX0NNICogcnMgKyBycyAqKiAyKSAvIDEwMDAuMAogICAgICAgIGlmIGRlYnVnOgogICAgICAgICAgICBwID0gbnAucGVyY2VudGlsZShkZXB0aHMsIFsxMCwgNTAsIDkwXSkKICAgICAgICAgICAgcHJpbnQoZiIgIFtkZWJ1Z10gY2VsbHM9e2xlbihkZXB0aHMpfSB0b3BfZGVwdGggcDEwLzUwLzkwPSIKICAgICAgICAgICAgICAgICAgZiJ7cFswXTouMGZ9L3twWzFdOi4wZn0ve3BbMl06LjBmfSBWPXtWX0w6LjBmfSBWZmxhdD17Vl9zaW1wbGU6LjBmfSIpCgogICAgaWYgdmlld3NfcG5nIG9yIGhlYXRtYXBfcG5nOgogICAgICAgIGltcG9ydCBtYXRwbG90bGliOyBtYXRwbG90bGliLnVzZSgiQWdnIik7IGltcG9ydCBtYXRwbG90bGliLnB5cGxvdCBhcyBwbHQKICAgICAgICBmcm9tIG1hdHBsb3RsaWIucGF0Y2hlcyBpbXBvcnQgQ2lyY2xlCiAgICAgICAgZGVwQSA9IC1LWzosIDJdOyByaG9BID0gbnAuaHlwb3QoS1s6LCAwXSwgS1s6LCAxXSkKICAgICAgICBjb2xBID0gX3dhbGxfZGVwdGgobnAuY2xpcChyaG9BLCAwLCBSX0NNKSkgLSBkZXBBICAgICAgICAgICMgYmlvY2hhciBiZW5lYXRoIGVhY2ggcHQKICAgICAgICBpc19iaW8gPSAocmhvQSA8PSBSX0NNKSAmIChjb2xBID4gQ09MX01JTikgICAgICAgICAgICAgICAgICMgYmlvY2hhciB2cyBraWxuIHN0cnVjdHVyZQoKICAgIGlmIHZpZXdzX3BuZzogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgMy1EIHJlY29uc3RydWN0aW9uICgyIHBhbmVscykKICAgICAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKDEsIDIsIGZpZ3NpemU9KDEyLCA2KSkKICAgICAgICBheFswXS5zY2F0dGVyKEtbfmlzX2JpbywgMF0sIEtbfmlzX2JpbywgMV0sIHM9MSwgYz0iI2NmYzdiNiIsIGxpbmV3aWR0aHM9MCkKICAgICAgICBpZiBpc19iaW8uYW55KCk6CiAgICAgICAgICAgIGF4WzBdLnNjYXR0ZXIoS1tpc19iaW8sIDBdLCBLW2lzX2JpbywgMV0sIHM9NSwgYz1jb2xBW2lzX2Jpb10sIGNtYXA9ImluZmVybm8iLCBsaW5ld2lkdGhzPTApCiAgICAgICAgYXhbMF0uYWRkX3BhdGNoKENpcmNsZSgoMCwgMCksIFJfQ00sIGZpbGw9RmFsc2UsIGVjPSIjQTk0RTI4IiwgbHc9MikpCiAgICAgICAgYXhbMF0uc2V0X3RpdGxlKCJUT1Ag4oCUIGJpb2NoYXIgKGNvbG91cikgaW5zaWRlIHRoZSBraWxuIChncmV5KSIpCiAgICAgICAgYXhbMV0uc2NhdHRlcihLW35pc19iaW8sIDBdLCBLW35pc19iaW8sIDJdLCBzPTEsIGM9IiNjZmM3YjYiLCBsaW5ld2lkdGhzPTApCiAgICAgICAgaWYgaXNfYmlvLmFueSgpOgogICAgICAgICAgICBheFsxXS5zY2F0dGVyKEtbaXNfYmlvLCAwXSwgS1tpc19iaW8sIDJdLCBzPTUsIGM9Y29sQVtpc19iaW9dLCBjbWFwPSJpbmZlcm5vIiwgbGluZXdpZHRocz0wKQogICAgICAgIGF4WzFdLnNldF90aXRsZSgiU0lERSDigJQgYmlvY2hhciBzaXRzIGluc2lkZSB0aGUgS29uLVRpa2kgY29uZSIpCiAgICAgICAgZm9yIGEgaW4gYXg6CiAgICAgICAgICAgIGEuc2V0X2FzcGVjdCgiZXF1YWwiLCAiYm94IikKICAgICAgICBmaWcudGlnaHRfbGF5b3V0KCk7IGZpZy5zYXZlZmlnKHZpZXdzX3BuZywgZHBpPTEzMCk7IHBsdC5jbG9zZShmaWcpCgogICAgaWYgaGVhdG1hcF9wbmcgYW5kIGNvbGdyaWQgaXMgbm90IE5vbmU6ICAgICAgICAgICAgICAgICAgICAgICAgIyBiaW9jaGFyIGhlYXRtYXAgb24gaXRzIG93bgogICAgICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoZmlnc2l6ZT0oNy42LCA2LjQpKQogICAgICAgIGltID0gYXguaW1zaG93KGNvbGdyaWQsIG9yaWdpbj0ibG93ZXIiLCBleHRlbnQ9Wy1SX0NNLCBSX0NNLCAtUl9DTSwgUl9DTV0sCiAgICAgICAgICAgICAgICAgICAgICAgY21hcD0iaW5mZXJubyIsIGludGVycG9sYXRpb249Im5lYXJlc3QiKQogICAgICAgIGZpZy5jb2xvcmJhcihpbSwgYXg9YXgsIGZyYWN0aW9uPTAuMDQ2LCBwYWQ9MC4wNCwgbGFiZWw9ImJpb2NoYXIgZGVwdGggKGNtKSIpCiAgICAgICAgYXguYWRkX3BhdGNoKENpcmNsZSgoMCwgMCksIFJfQ00sIGZpbGw9RmFsc2UsIGVjPSIjQTk0RTI4IiwgbHc9MS44KSkKICAgICAgICBheC5zZXRfdGl0bGUoIkJpb2NoYXIgZGVwdGggaGVhdG1hcCAgwrcgIHZvbHVtZSA9IHN1bSBvZiB0aGlzIikKICAgICAgICBheC5zZXRfYXNwZWN0KCJlcXVhbCIsICJib3giKQogICAgICAgIGZpZy50aWdodF9sYXlvdXQoKTsgZmlnLnNhdmVmaWcoaGVhdG1hcF9wbmcsIGRwaT0xMzApOyBwbHQuY2xvc2UoZmlnKQoKICAgIHJldHVybiB7InZvbHVtZV9MIjogVl9MLCAidm9sdW1lX0xfZmxhdGZpbGwiOiBWX3NpbXBsZSwgImZpbGxfaGVpZ2h0X2NtIjogaF9maWxsLAogICAgICAgICAgICAiZmlsbF9wY3QiOiAxMDAgKiBWX0wgLyBWX2Z1bGwsICJ3ZWlnaHRfa2ciOiBERU5TSVRZICogVl9MLAogICAgICAgICAgICAibWVhc3VyZWRfcmltX3VuaXRzIjogcl91bml0cywgInNjYWxlX2NtX3Blcl91bml0Ijogc30KCgpkZWYgZXN0aW1hdGUocGx5X3BhdGgsIHJpbV9yYWRpdXNfY209Ul9DTSwgdmlld3NfcG5nPU5vbmUsIGhlYXRtYXBfcG5nPU5vbmUpOgogICAgaW1wb3J0IG9wZW4zZCBhcyBvM2QKICAgIHBjZCA9IG8zZC5pby5yZWFkX3BvaW50X2Nsb3VkKHBseV9wYXRoKQogICAgaWYgbGVuKHBjZC5wb2ludHMpID09IDA6CiAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiZW1wdHkgcG9pbnQgY2xvdWQiKQogICAgcmV0dXJuIGVzdGltYXRlX3BvaW50cyhucC5hc2FycmF5KHBjZC5wb2ludHMpLCByaW1fcmFkaXVzX2NtLCB2aWV3c19wbmcsIGhlYXRtYXBfcG5nKQoKCmRlZiBfcHJpbnQocmVzKToKICAgIHByaW50KCI9IiAqIDQ2KQogICAgcHJpbnQoZiIgIEJJT0NIQVIgVk9MVU1FIChpbnRlZ3JhdGVkKSA6IHtyZXNbJ3ZvbHVtZV9MJ106Ni4wZn0gTCIpCiAgICBwcmludChmIiAgY3Jvc3MtY2hlY2sgKGZsYXQgZmlsbCkgICAgIDoge3Jlc1sndm9sdW1lX0xfZmxhdGZpbGwnXTo2LjBmfSBMIikKICAgIHByaW50KGYiICBmaWxsIGhlaWdodCAvIGZpbGwgJSAgICAgICAgOiB7cmVzWydmaWxsX2hlaWdodF9jbSddOi4wZn0gY20gLyB7cmVzWydmaWxsX3BjdCddOi4wZn0lIikKICAgIHByaW50KGYiICBhcHByb3ggd2VpZ2h0ICh+MC4yNSBrZy9MKSAgOiB7cmVzWyd3ZWlnaHRfa2cnXTo2LjBmfSBrZyIpCiAgICBwcmludChmIiAgc2NhbGUgICAgICAgICAgICAgICAgICAgICAgIDoge3Jlc1snc2NhbGVfY21fcGVyX3VuaXQnXTouNGZ9IGNtL3VuaXQiKQogICAgcHJpbnQoIj0iICogNDYpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIGlmIGxlbihzeXMuYXJndikgPCAyOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoX19kb2NfXykKICAgIHBseSA9IHN5cy5hcmd2WzFdCiAgICByaW0gPSBmbG9hdChzeXMuYXJndlsyXSkgaWYgbGVuKHN5cy5hcmd2KSA+IDIgZWxzZSBSX0NNCiAgICBwbmcgPSBzeXMuYXJndlszXSBpZiBsZW4oc3lzLmFyZ3YpID4gMyBlbHNlIE5vbmUKICAgIF9wcmludChlc3RpbWF0ZShwbHksIHJpbSwgcG5nKSkK").decode("utf-8"))
from estimate_volume import estimate_points
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map
import gradio as gr

print("loading VGGT model (once)...")
MODEL = VGGT.from_pretrained("facebook/VGGT-1B").to("cuda").eval()

def extract_frames(video, n=32, out="frames"):
    if os.path.exists(out): shutil.rmtree(out)
    os.makedirs(out)
    cap = cv2.VideoCapture(video); total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if total <= 0:
        total = 0
        while cap.grab(): total += 1
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    slot = total / n; k = 0
    for i in range(n):
        lo, hi = int(i * slot), int((i + 1) * slot); best = None
        for idx in np.linspace(lo, max(lo, hi - 1), 5).astype(int):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx)); ok, fr = cap.read()
            if not ok: continue
            sc = cv2.Laplacian(cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
            if best is None or sc > best[1]: best = (idx, sc, fr)
        if best:
            cv2.imwrite(f"{out}/f_{k:03d}.jpg", best[2], [cv2.IMWRITE_JPEG_QUALITY, 95]); k += 1
    cap.release(); return sorted(glob.glob(f"{out}/*.jpg"))

def reconstruct(paths):
    dt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    imgs = load_and_preprocess_images(paths).to("cuda")
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=dt):
        pred = MODEL(imgs)
    def gk(d, *ks):
        for kk in ks:
            if kk in d: return d[kk]
        raise KeyError(ks)
    extr, intr = pose_encoding_to_extri_intri(gk(pred, "pose_enc"), imgs.shape[-2:])
    depth = gk(pred, "depth", "depth_map"); conf = gk(pred, "depth_conf", "point_conf", "depth_confidence")
    w = np.asarray(unproject_depth_map_to_point_map(depth.squeeze(0), extr.squeeze(0), intr.squeeze(0))).reshape(-1, 3)
    conf = conf.squeeze(0).float().cpu().numpy().reshape(-1)
    w = w[(conf >= np.quantile(conf, 0.5)) & np.isfinite(w).all(1)]
    if len(w) > 300000:
        w = w[np.random.default_rng(0).choice(len(w), 300000, replace=False)]
    return w

def process(video):
    if not video:
        return "<p>Please upload a kiln video.</p>", None, None
    try:
        paths = extract_frames(video)
        world = reconstruct(paths)
        res = estimate_points(world, rim_radius_cm=75.0, views_png="views.png", heatmap_png="heatmap.png")
    except Exception as e:
        return f"<p style='color:#b00'>Could not process this video: {e}</p>", None, None
    V, Vf = res["volume_L"], res["volume_L_flatfill"]
    gap = abs(V - Vf) / max(V, 1) * 100; fill = max(0, min(100, res["fill_pct"]))
    trust = (f"<span style='color:#2e7d33'>&#10003; estimates agree ({gap:.0f}%)</span>" if gap < 8
             else f"<span style='color:#c60'>&#9888; estimates disagree {gap:.0f}% &mdash; re-shoot per the SOP</span>")
    html = f"""<div style="font-family:system-ui,Segoe UI,Arial">
      <div style="font-size:13px;letter-spacing:1px;color:#888">ESTIMATED BIOCHAR VOLUME</div>
      <div style="font-size:54px;font-weight:800;color:#2e7d33;line-height:1">{V:,.0f} L</div>
      <div style="color:#555;margin-top:4px">&#8776; {V*0.25:,.0f} kg &middot; {V/1000:.2f} m&sup3;</div>
      <div style="margin-top:12px;font-size:13px;color:#888">FILL &mdash; {fill:.0f}% of a ~998 L kiln (height {res['fill_height_cm']:.0f} cm)</div>
      <div style="height:16px;background:#eee;border-radius:9px;overflow:hidden;margin-top:4px">
        <div style="height:100%;width:{fill:.0f}%;background:linear-gradient(90deg,#2d7ef7,#4fe08a)"></div></div>
      <div style="margin-top:12px">cross-check {Vf:,.0f} L &nbsp; {trust}</div>
    </div>"""
    return html, "views.png", "heatmap.png"

demo = gr.Interface(
    fn=process,
    inputs=gr.Video(label="Upload a slow-orbit kiln video"),
    outputs=[gr.HTML(label="Result"),
             gr.Image(label="3-D reconstruction (top = rim disk, side = cone)"),
             gr.Image(label="Biochar depth heatmap (volume = sum)")],
    title="Kon-Tiki Biochar Volume - Video Dashboard",
    description="Upload a slow orbit video of the biochar-filled kiln. It extracts frames, reconstructs the 3-D shape on GPU, and returns the biochar volume. Follow the capture SOP for best accuracy.")
print("launching dashboard - a public https://....gradio.live link will appear below")
demo.launch(share=True)


### Notes
- The **public link** works from any device (phone/laptop) for ~72 h while this cell runs.
- Keep this Colab tab open; closing it stops the dashboard. Re-run the cell to restart.
- For an **always-on** dashboard (no Colab), deploy the same app to a **GPU host**
  (Hugging Face Spaces GPU / Modal) — ask and I'll provide `gradio_app.py` + steps.
